In [0]:
# =============================================================================
# Notebook  : 02b_map_06_accounts_attributes
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_06_accounts_attributes
# Spec Ref  : §1.9 accounts_attributes (account_id, is_paying, is_excluded, mrr)
# Purpose   : Create one attributes row for EVERY account.
#             Spec DQ gate: SELECT COUNT(*) FROM accounts_all a
#               LEFT JOIN accounts_attributes aa ON a.id = aa.account_id
#               WHERE aa.account_id IS NULL  ← must return 0
# Run after : map_02 (accounts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
print(f"=== Map 06: accounts_attributes ===  customer={customer_id}")

In [0]:
# CELL 3
create_map_table(
    f"{sv}.accounts_attributes",
    """
        account_id   STRING NOT NULL,
        tenant       BIGINT,
        is_paying    BOOLEAN,
        is_excluded  BOOLEAN,
        mrr          DOUBLE
    """,
    cluster_by="account_id"
)

In [0]:
# CELL 4 -- Full rebuild: one row per account
# is_paying, is_excluded, mrr default to safe values.
# Will be updated by downstream enrichment/scoring stages.

# Safe drop in case target exists as a VIEW
safe_drop_view(f"{sv}.accounts_attributes")

spark.sql(f"""
    CREATE OR REPLACE TABLE {sv}.accounts_attributes
    USING DELTA
    CLUSTER BY (account_id)
    {DELTA_TBLPROPS_MAP}
    AS
    SELECT
        id      AS account_id,
        tenant,
        false   AS is_paying,
        false   AS is_excluded,
        0.0     AS mrr
    FROM {sv}.accounts_all
""")

In [0]:
# CELL 5 -- Spec DQ gate: every account must have an attributes row
uncovered = spark.sql(f"""
    SELECT COUNT(*) FROM {sv}.accounts_all a
    LEFT JOIN {sv}.accounts_attributes aa ON a.id = aa.account_id
    WHERE aa.account_id IS NULL
""").collect()[0][0]

n = cnt(f"{sv}.accounts_attributes")
print(f"  accounts_attributes: {n:,} rows")
status = '✅ PASS' if uncovered == 0 else '❌ FAIL'
print(f"  Uncovered accounts: {uncovered}  (spec gate = 0)  {status}")
dbutils.notebook.exit("Success")